In [1]:
import numpy as np

In [2]:
class Tensor(object):
    def __init__(self, data, creators=None, creation_op=None):
        self.data = np.array(data)
        self.creators = creators
        self.creation_op = creation_op
        self.grad = None

    def backward(self, grad):
        self.grad = grad

        if (self.creation_op == 'add'):
            self.creators[0].backward(grad)
            self.creators[1].backward(grad)

    def __add__(self, other):
        return Tensor(self.data + other.data,
                      creators=[self, other],
                      creation_op='add')

    def __repr__(self):
        return str(self.data.__repr__())

    def __str__(self):
        return str(self.data.__str__())

    def dump(self):
        print(self.creators, " | ", self.creation_op, " | ", self.grad)

In [3]:
x = Tensor([1, 2, 3])
y = Tensor([4, 5, 6])
z = x + y
z.dump()

[array([1, 2, 3]), array([4, 5, 6])]  |  add  |  None


In [4]:
z.backward(Tensor(np.array([5, 6, 7])))
z.dump()

[array([1, 2, 3]), array([4, 5, 6])]  |  add  |  [5 6 7]


In [5]:
a = Tensor([1, 2, 3, 4, 5])
b = Tensor([2, 2, 2, 2, 2])
c = Tensor([5, 4, 3, 2, 1])
d = Tensor([-1, -2, -3, -4, -5])
e = a + b
f = c + d
g = e + f
g.backward(Tensor(np.array([1, 1, 1, 1, 1])))
a.dump()

None  |  None  |  [1 1 1 1 1]


In [6]:
# Текущая версия класса Tensor поддерживает обратное распространение градиентов только один раз
# для одной переменной, но иногда во время прямого прохода мы будем использовать тот же тензор
# несколько раз, и поэтому несколько частей графа будут распространять
# градиенты обратно в тот же Тензор.

a = Tensor([1, 2, 3, 4, 5])
b = Tensor([2, 2, 2, 2, 2])
c = Tensor([5, 4, 3, 2, 1])

d = a + b
e = b + e
f = d + e

f.backward(Tensor([1, 1, 1, 1, 1]))

print(
    b.grad.data,
    b.grad.data == [2, 2, 2, 2, 2]
)

[1 1 1 1 1] [False False False False False]


# Design basic Tensor class

In [121]:
import numpy as np


class Tensor(object):
    def __init__(self, data, autograd=False, parents=None, creation_op=None, id=None):
        self.data = np.array(data)
        self.autograd = autograd
        self.parents = parents
        self.children = {}
        self.creation_op = creation_op
        self.grad = None
        if (id is None): id = np.random.randint(1000)
        self.id = id

        if (parents is not None):
            for parent in parents:
                if (self.id not in parent.children):
                    parent.children[self] = 1
                else:
                    parent.children[self] += 1

    def all_grads_propagated(self):
        for _, grads_count in self.children.items():
            if (grads_count != 0): return False
        return True

    def __add__(self, other):
        if (self.autograd and other.autograd):
            return Tensor(self.data + other.data,
                          autograd=True,
                          parents=[self, other],
                          creation_op="+")
        return Tensor(self.data + other.data)

    def __sub__(self, other):
        if (self.autograd and other.autograd):
            return Tensor(self.data - other.data,
                          autograd=True,
                          parents=[self, other],
                          creation_op="-")
        return Tensor(self.data - other.data)

    def __mul__(self, other):
        if (self.autograd and other.autograd):
            return Tensor(self.data * other.data,
                          autograd=True,
                          parents=[self, other],
                          creation_op="*")
        return Tensor(self.data * other.data)

    def sum(self, dim):
        if (self.autograd):
            return Tensor(self.data.sum(dim),
                          autograd=True,
                          parents=[self],
                          creation_op="sum_" + str(dim))
        return Tensor(self.data.sum(dim))

    def __neg__(self):
        if (self.autograd):
            return Tensor(self.data * -1,
                          autograd=True,
                          parents=[self],
                          creation_op="neg")
        return Tensor(self.data * -1)

    def __repr__(self):
        return str('Tensor(' + self.id.__repr__() + ') Data: ' + self.data.__str__())

    def __str__(self):
        return self.__repr__()

    def expand(self, dim, copies):
        trans_cmd = list(range(0, len(self.data.shape)))
        trans_cmd.insert(dim, len(self.data.shape))

        new_shape = list(self.data.shape) + [copies]

        new_data = self.data.repeat(copies).reshape(new_shape)
        new_data = new_data.transpose(trans_cmd)

        if (self.autograd):
            return Tensor(new_data,
                          autograd=True,
                          parents=[self],
                          creation_op="expand_" + str(dim))
        return Tensor(new_data)

    def transpose(self):
        if (self.autograd):
            return Tensor(self.data.transpose(),
                          autograd=True,
                          parents=[self],
                          creation_op="T")
        return Tensor(self.data.transpose())

    def __matmul__(self, other):
        if (self.autograd):
            return Tensor(self.data @ other.data,
                          autograd=True,
                          parents=[self, other],
                          creation_op="mm")
        return Tensor(self.data @ other.data)

    def mm(self, other):
        return self.__matmul__(other)

    def backward(self, grad=None, grad_origin=None):
        if (self.autograd):
            if (grad == None):
                grad = Tensor(np.ones_like(self.data))

            if (grad_origin is not None):
                if (self.children[grad_origin] == 0):
                    raise Exception("cannot backprop more than once")
                else:
                    self.children[grad_origin] -= 1

            if (self.grad is None):
                self.grad = grad
            else:
                self.grad += grad

            if ((self.parents is not None) and (self.all_grads_propagated() or grad_origin is None)):
                if (self.creation_op == "+"):
                    self.parents[0].backward(self.grad, grad_origin=self)
                    self.parents[1].backward(self.grad, grad_origin=self)

                if (self.creation_op == "neg"):
                    self.parents[0].backward(self.grad.__neg__())

                if (self.creation_op == '-'):
                    self.parents[0].backward(self.grad, grad_origin=self)
                    self.parents[1].backward(self.grad.__neg__(), grad_origin=self)

                if (self.creation_op == '*'):
                    self.parents[0].backward(self.grad * self.parents[1], grad_origin=self)
                    self.parents[1].backward(self.grad * self.parents[0], grad_origin=self)

                if (self.creation_op == 'mm'):
                    activation = self.parents[0]  # usually an activation function
                    weights = self.parents[1]  # usually a weights matrix
                    activation.backward(self.grad.mm(weights.transpose()))
                    weights.backward(self.grad.transpose().mm(activation).transpose())

                if (self.creation_op == 'T'):
                    self.parents[0].backward(self.grad.transpose())

                if ("sum" in self.creation_op):
                    dim = int(self.creation_op.split("_")[1])
                    ds = self.parents[0].data.shape[dim]
                    self.parents[0].backward(self.grad.expand(dim, ds))

                if ("expand" in self.creation_op):
                    dim = int(self.creation_op.split("_")[1])
                    self.parents[0].backward(self.grad.sum(dim))

    def dump(self):
        print("=" * 50)
        print(f"Tensor ID: {self.id}")
        print(f"Data: {self.data}")
        print(f"Autograd: {self.autograd}")
        print(f"Gradient: {self.grad}")
        print(f"Creation Operation: {self.creation_op}")

        print("\nParents:")
        if self.parents is not None:
            for i, parent in enumerate(self.parents):
                print(f"ID: {parent.id}: {parent.data}")
        else:
            print("  None")

        print("\nChildren:")
        if self.children:
            for child_id, count in self.children.items():
                print(f"  Child ID: {child_id.id}, Count: {count}")
        else:
            print("  No children")

In [8]:
# Проверяем, работает ли автоград

a = Tensor([1, 2, 3, 4, 5], id="a", autograd=True)
b = Tensor([2, 2, 2, 2, 2], id="b", autograd=True)
c = Tensor([5, 4, 3, 2, 1], id="c", autograd=True)

d = a + b
e = b + c
f = d + e

f.backward(Tensor([1, 1, 5, 1, 1]))

print(b.grad.data == [2, 2, 10, 2, 2])
b.dump()

[ True  True  True  True  True]
Tensor ID: b
Data: [2 2 2 2 2]
Autograd: True
Gradient: Tensor(992) Data: [ 2  2 10  2  2]
Creation Operation: None

Parents:
  None

Children:
  Child ID: 2, Count: 0
  Child ID: 926, Count: 0


In [9]:
# test __add__

a = Tensor([1, 2, 3])
b = Tensor([4, 5, 6])
a + b

Tensor(847) Data: [5 7 9]

In [10]:
# test __neg__

a = Tensor([1, 2, 3])
-a

Tensor(475) Data: [-1 -2 -3]

In [11]:
# test __sub__

a = Tensor([1, 2, 3])
b = Tensor([4, 5, 6])
(a - b)

Tensor(295) Data: [-3 -3 -3]

In [12]:
# test __mul__

a = Tensor([1, 2, 3])
b = Tensor([4, 5, 6])

(a * b)

Tensor(301) Data: [ 4 10 18]

In [122]:
# test __matmul__

a = Tensor([1, 2, 3])
b = Tensor([4, 5, 6])

expected_value = np.array([1, 2, 3]) @ np.array([4, 5, 6])
print(f"Expected: {expected_value}")

print(a @ b)
print(a.mm(b))

Expected: 32
Tensor(771) Data: 32
Tensor(821) Data: 32


In [14]:
# test transpose

a = Tensor([[1, 2, 3], [4, 5, 6], [7, 8, 9]])
a.transpose()

Tensor(560) Data: [[1 4 7]
 [2 5 8]
 [3 6 9]]

In [27]:
# test sum

a = Tensor([1, 2, 3])
print(a.sum(0))
print()

b = Tensor([[1, 2, 3], [4, 5, 6]])
print(b.sum(0))
print(b.sum(1))
print()

c = Tensor([[1, 2, 3], [4, 5, 6], [7, 8, 9]])
print(c.sum(0))
print(c.sum(1))
c.data

Tensor(794) Data: 6

Tensor(262) Data: [5 7 9]
Tensor(788) Data: [ 6 15]

Tensor(123) Data: [12 15 18]
Tensor(860) Data: [ 6 15 24]


array([[1, 2, 3],
       [4, 5, 6],
       [7, 8, 9]])

In [22]:
# test sum

a = Tensor([1, 2, 3])
print(a.expand(0, 2))

b = Tensor([[1, 2, 3], [4, 5, 6], [7, 8, 9]])
print(b.expand(0, 2))
print(b.expand(1, 2))
print(b.expand(2, 2))

Tensor(672) Data: [[1 2 3]
 [1 2 3]]
Tensor(410) Data: [[[1 2 3]
  [4 5 6]
  [7 8 9]]

 [[1 2 3]
  [4 5 6]
  [7 8 9]]]
Tensor(946) Data: [[[1 2 3]
  [1 2 3]]

 [[4 5 6]
  [4 5 6]]

 [[7 8 9]
  [7 8 9]]]
Tensor(796) Data: [[[1 1]
  [2 2]
  [3 3]]

 [[4 4]
  [5 5]
  [6 6]]

 [[7 7]
  [8 8]
  [9 9]]]


# Using Tensor class to train NN

In [161]:
import numpy as np

np.random.seed(0)

dataset = np.array([
    [0, 0, 0],
    [0, 1, 1],
    [1, 0, 0],
    [1, 1, 1],
])

X_train = dataset[:, :2]  # (4, 2)
Y_train = dataset[:, 2:]  # (4, 1)

hidden_size = 3
epochs = 10
alpha = 0.1

weights_0_1 = np.random.rand(X_train.shape[1], hidden_size)  # (2, 3)
weights_1_2 = np.random.rand(hidden_size, Y_train.shape[1])  # (3, 1)

for i in range(epochs):
    # Прямой проход
    layer_1 = X_train @ weights_0_1  # (4, 3)
    layer_2 = layer_1 @ weights_1_2  # (4, 1)

    # На сколько мимо от цели
    diff = layer_2 - Y_train  # (4, 1)

    # Квадрат убирает знак и усиливает ошибку
    # И суммируем ее чтобы использовать как метрику
    loss = (diff ** 2).sum(0)  # (1,)

    # Обратное распространение
    # 1. Считаем градиенты между выходом и внутренем слоем
    layer_1_grad = diff @ weights_1_2.T  # (4, 1) @ (1, 10) = (4, 10)
    # 2. Считаем на сколько нужно изменить вес между 1 и 2
    weights_1_2_delta = layer_1.T @ diff  # (3, 4) @ (4, 1) = (3, 1)
    # 3. Считаем на сколько нужно изменить вес между 0 и 1
    weights_0_1_delta = X_train.T @ layer_1_grad  # (3, 4) @ (4, 3) = (3, 3)

    # Обновляем веса с учетом шага обучения
    weights_1_2 -= weights_1_2_delta * alpha
    weights_0_1 -= weights_0_1_delta * alpha

    print(loss[0])


5.066439994622395
0.4959907791902342
0.4180671892167177
0.35298133007809646
0.2972549636567377
0.2492326038163328
0.20785392075862477
0.17231260916265176
0.14193744536652986
0.11613979792168384


In [136]:
import numpy as np

np.random.seed(42)

dataset = np.array([
    [0, 0, 0, 0],
    [0, 0, 1, 0],
    [0, 1, 0, 0],
    [0, 1, 1, 1],
    [1, 0, 0, 0],
    [1, 0, 1, 1],
    [1, 1, 0, 1],
    [1, 1, 1, 1],
])

X_train = dataset[:, :3]  # (8, 3)
Y_train = dataset[:, 3:]  # (8, 1)

hidden_size = 10
epochs = 50
alpha = 0.01

weights_0_1 = np.random.rand(X_train.shape[1], hidden_size)  # (3, 10)
weights_1_2 = np.random.rand(hidden_size, Y_train.shape[1])  # (10, 1)

for i in range(epochs):
    # Прямой проход
    layer_1 = X_train @ weights_0_1  # (8, 10)
    layer_2 = layer_1 @ weights_1_2  # (8, 1)

    # На сколько мимо от цели
    diff = layer_2 - Y_train  # (8, 1)

    # Квадрат убирает знак и усиливает ошибку
    # И суммируем ее чтобы использовать как метрику
    loss = (diff ** 2).sum(0)

    # Обратное распространение
    # 1. Считаем градиенты между выходом и внутренем слоем
    layer_1_grad = diff @ weights_1_2.T  # (8, 1) @ (1, 10) = (8, 10)
    # 2. Считаем на сколько нужно изменить вес между 1 и 2
    weights_1_2_delta = layer_1.T @ diff  # (10, 8) @ (8, 1) = (10, 1)
    # 3. Считаем на сколько нужно изменить вес между 0 и 1
    weights_0_1_delta = X_train.T @ layer_1_grad  # (3, 8) @ (8, 10) = (3, 10)

    # Обновляем веса с учетом шага обучения
    weights_1_2 -= weights_1_2_delta * alpha
    weights_0_1 -= weights_0_1_delta * alpha

    if (i % 5 == 0 or i == epochs - 1):
        print(str(i) + ": " + str(round(loss[0], 4)))


0: 60.259
5: 1.5034
10: 1.1035
15: 0.894
20: 0.7789
25: 0.7141
30: 0.677
35: 0.6556
40: 0.643
45: 0.6357
49: 0.632


In [158]:
import numpy as np

np.random.seed(42)

dataset = np.array([
    [0, 0, 0, 0],
    [0, 0, 1, 0],
    [0, 1, 0, 0],
    [0, 1, 1, 1],
    [1, 0, 0, 0],
    [1, 0, 1, 1],
    [1, 1, 0, 1],
    [1, 1, 1, 1],
])

X_train = Tensor(dataset[:, :3], autograd=True)  # (8, 3)
Y_train = Tensor(dataset[:, 3:], autograd=True)  # (8, 1)

hidden_size = 10
epochs = 10
alpha = 0.01

weights = list()
weights.append(Tensor(np.random.rand(3, hidden_size), autograd=True))
weights.append(Tensor(np.random.rand(hidden_size, 1), autograd=True))

for i in range(epochs):

    predict = X_train @ weights[0] @ weights[1]
    loss = ((predict - Y_train) * (predict - Y_train)).sum(0)

    loss.backward(Tensor(np.ones_like(loss.data)))

    for weight in weights:
        weight.data -= weight.grad.data * alpha
        weight.grad.data *= 0

    print(str(i) + ": " + str(round(loss.data[0], 4)))

0: 46.2668
1: 4.7614
2: 0.7045
3: 0.6878
4: 0.6797
5: 0.6727
6: 0.6666
7: 0.6614
8: 0.6568
9: 0.6528


# Auto optimization

In [163]:
class SGD(object):
    def __init__(self, weights, alpha=0.01):
        self.weights = weights
        self.alpha = alpha

    def step(self, zero=True):
        for weight in self.weights:
            weight.data -= weight.grad.data * self.alpha
            if (zero):
                weight.grad.data *= 0

    def zero(self):
        for weight in self.weights:
            weight.grad.data *= 0


In [164]:
import numpy as np

np.random.seed(0)

dataset = np.array([
    [0, 0, 0],
    [0, 1, 1],
    [1, 0, 0],
    [1, 1, 1],
])

X_train = dataset[:, :2]  # (4, 2)
Y_train = dataset[:, 2:]  # (4, 1)

data = Tensor(X_train, autograd=True)
target = Tensor(Y_train, autograd=True)

w = list()
w.append(Tensor(np.random.rand(2, 3), autograd=True))
w.append(Tensor(np.random.rand(3, 1), autograd=True))

optim = SGD(weights=w, alpha=0.1)

for i in range(10):
    pred = data @ w[0] @ w[1]
    loss = ((pred - target) * (pred - target)).sum(0)
    loss.backward(Tensor(np.ones_like(loss.data)))
    optim.step()

    print(loss.data[0])


0.5812830360381691
0.4898814918030678
0.4137511099699248
0.34489412208720704
0.2821012414527573
0.2254484048015707
0.17538852854551776
0.13242309965665466
0.09682768724516845
0.0684936057498309
